# Chunking & semantic clustering — FoodScholar Abstracts (bge-large-en-v1.5)

Pipeline over `chunks/abstracts/curated_abstracts.csv`:

1. **§1–3** — load the `BAAI/bge-large-en-v1.5` tokenizer, the abstracts CSV, and a sentence-level 512/64-overlap split estimator (mirror of `chunking_pipeline_guides_overlap.ipynb`, no HybridChunker).
2. **§4a** — count, per abstract: token count, sentence count, `needs_split` (>512 tokens), `estimated_chunks`.
3. **§4b** — actually split abstracts **> 512 tokens** into **512-token chunks with 64-token overlap** and write them to a new CSV in the **corpus format** of `corpus_top_journal.csv` (`chunk_id, chunk_text, type, chunk_metadata`).
4. **§5–6** — summary statistics + save the per-abstract counts.
5. **§7–11** — semantic clustering of the chunk corpus: normalized `BAAI/bge-large-en-v1.5` embeddings (GPU-aware, memory-mapped & reusable), scalable `MiniBatchKMeans` partition into a configurable number of semantically similar, non-overlapping clusters, per-cluster CSVs (all original columns preserved), cluster statistics, and verification that every input row ends up in exactly one output file.

In [ ]:
# --- Environment: HuggingFace cache (same as the guides pipeline) ---
import os
from pathlib import Path

CACHE_ROOT = Path(".cache")
HF_HOME_DIR = CACHE_ROOT / "huggingface"
HF_HUB_CACHE_DIR = HF_HOME_DIR / "hub"
HF_TRANSFORMERS_CACHE_DIR = HF_HOME_DIR / "transformers"

for path in [CACHE_ROOT, HF_HOME_DIR, HF_HUB_CACHE_DIR, HF_TRANSFORMERS_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

os.environ.update(
    {
        "XDG_CACHE_HOME": str(Path(CACHE_ROOT)),
        "HF_HOME": str(HF_HOME_DIR),
        "HF_HUB_CACHE": str(HF_HUB_CACHE_DIR),
        "HUGGINGFACE_HUB_CACHE": str(HF_HUB_CACHE_DIR),
        "TRANSFORMERS_CACHE": str(HF_TRANSFORMERS_CACHE_DIR),
        "HF_HUB_DISABLE_SYMLINKS_WARNING": "1",
    }
)

## 1. Load the embedding tokenizer (BAAI/bge-large-en-v1.5)

In [ ]:
from transformers import AutoTokenizer

# Same embedding model used by the guides / textbooks pipelines.
EMBED_MODEL_ID = "BAAI/bge-large-en-v1.5"

hf_tokenizer = AutoTokenizer.from_pretrained(
    EMBED_MODEL_ID,
    use_fast=False,   # same as the guides pipeline
)

# Sanity check
print(f"✅ Loaded tokenizer: {EMBED_MODEL_ID}")
print("   sample tokens:", hf_tokenizer.tokenize(
    "Whole-grain intake is associated with lower risk of type 2 diabetes."
)[:12])

## 2. Load the abstracts CSV

In [ ]:
import pandas as pd

ABSTRACTS_CSV = "../data/original/abstracts/curated_abstracts.csv"
abstracts_df = pd.read_csv(ABSTRACTS_CSV, low_memory=False)

print(f"✅ Loaded: {ABSTRACTS_CSV}")
print(f"   shape: {abstracts_df.shape}")
print("   columns:", ", ".join(abstracts_df.columns))
print()
print("abstract_quality counts:")
print(abstracts_df["abstract_quality"].value_counts(dropna=False))
print()
print(f"null abstracts : {abstracts_df['abstract'].isna().sum()}")
print(f"empty abstracts: {(abstracts_df['abstract'].astype(str).str.strip() == '').sum()}")

## 3. Split-need estimator (mirror of the guides strategy, no HybridChunker)

Re-implements the sliding window from `build_overlapping_chunks()` in `chunking_pipeline_guides_overlap.ipynb` for a **single group** (abstracts have no headings):

- sentences play the role of the guides' *fine chunks*,
- windows are expanded to at most `MAX_TOKENS = 512`,
- the next window starts so that the previous window's tail keeps `OVERLAP = 64` tokens,
- oversized single sentences are emitted as one chunk (same as the guides' fine-chunk fallback).

The function only **counts** chunks — it never builds or saves them.

In [ ]:
from nltk.tokenize import sent_tokenize

# ── Same parameters as the guides pipeline ──
MAX_TOKENS = 512   # max tokens per chunk
OVERLAP    = 64    # overlap tokens between consecutive chunks

def count_tokens(text) -> int:
    """Token count using the BAAI/bge-large-en-v1.5 tokenizer (same as guides)."""
    if not isinstance(text, str) or not text.strip():
        return 0
    return len(hf_tokenizer.tokenize(text))

def split_into_sentences(text) -> list[str]:
    """Plain-text sentence split (nltk punkt). No HybridChunker — raw text only."""
    if not isinstance(text, str) or not text.strip():
        return []
    return [s.strip() for s in sent_tokenize(text) if s.strip()]

def estimate_split(sentences: list[str]) -> dict:
    """
    Simulate the guides' overlapping split (512 tokens, 64 overlap) over sentences.
    Mirrors build_overlapping_chunks() from chunking_pipeline_guides_overlap.ipynb
    for a single group (no headings) — sentences act as the fine chunks.
    Does NOT materialize chunks — only counts them.
    Returns: {num_chunks, chunk_tokens, overlaps}
    """
    n = len(sentences)
    if n == 0:
        return {"num_chunks": 0, "chunk_tokens": [], "overlaps": []}

    token_counts = [count_tokens(s) for s in sentences]
    cumsum = [0] * (n + 1)
    for i in range(n):
        cumsum[i + 1] = cumsum[i] + token_counts[i]

    num_chunks = 0
    chunk_tokens: list[int] = []
    overlaps: list[int] = []

    start = 0
    while start < n:
        # All remaining sentences fit in one chunk → emit once and stop.
        if cumsum[n] - cumsum[start] <= MAX_TOKENS:
            num_chunks += 1
            chunk_tokens.append(cumsum[n] - cumsum[start])
            break

        # Expand end as far as possible within MAX_TOKENS.
        end = start
        while end < n and (cumsum[end + 1] - cumsum[start]) <= MAX_TOKENS:
            end += 1

        if end == start:
            # A single sentence alone exceeds MAX_TOKENS → emitted as one chunk
            # (same fallback as the guides pipeline for oversized fine chunks).
            num_chunks += 1
            chunk_tokens.append(token_counts[start])
            start += 1
            continue

        # Current chunk = sentences [start, end).
        num_chunks += 1
        chunk_tokens.append(cumsum[end] - cumsum[start])

        # Advance start for ~OVERLAP tokens (exact logic from the guides notebook):
        # find the largest k whose tail (tokens of [k, end)) still covers OVERLAP.
        new_start = start
        for k in range(start + 1, end + 1):
            tail = cumsum[end] - cumsum[k]
            if tail >= OVERLAP:
                new_start = k
            else:
                break
        start = max(new_start, start + 1)
        overlaps.append(cumsum[end] - cumsum[start])

    return {"num_chunks": num_chunks, "chunk_tokens": chunk_tokens, "overlaps": overlaps}

# ── Quick self-test on a synthetic abstract ──
_test = (
    "Dietary guidelines provide evidence-based recommendations on healthy eating. "
    "They translate nutrient targets into food-based advice for the general population. "
) * 20
_sents = split_into_sentences(_test)
print(f"✅ helpers ready — test abstract: {count_tokens(_test)} tokens, "
      f"{len(_sents)} sentences → {estimate_split(_sents)['num_chunks']} chunks @ {MAX_TOKENS}/{OVERLAP}")

## 4a. Count split need for every abstract

For each row: token count, sentence count, `needs_split` (token count > 512) and `estimated_chunks` (number of 512/64-overlap chunks the abstract would be split into).

> ⚠️ This cell only **counts**. The actual split + CSV export happens in **4b** below.

In [ ]:
from tqdm.auto import tqdm

# ── Optional smoke-test limit (set to None to process ALL rows) ──
LIMIT = None   # e.g. 5000 for a quick test run

work = abstracts_df.head(LIMIT) if LIMIT else abstracts_df

records = []
for _, row in tqdm(work.iterrows(), total=len(work), desc="Counting split need"):
    text = row["abstract"] if isinstance(row["abstract"], str) else ""
    token_count = count_tokens(text)
    sentences = split_into_sentences(text)
    est = estimate_split(sentences)
    records.append({
        "paperId":             row.get("paperId"),
        "title":               row.get("title"),
        "year":                row.get("year"),
        "abstract_quality":    row.get("abstract_quality"),
        "abstract_token_count": token_count,
        "num_sentences":       len(sentences),
        "needs_split":         token_count > MAX_TOKENS,
        "estimated_chunks":    est["num_chunks"],
    })

counts_df = pd.DataFrame(records)
print(f"✅ Counted {len(counts_df):,} abstracts")
counts_df.head(10)

## 4b. Split abstracts > 512 tokens into 512/64 chunks → new CSV

For every abstract with `token_count > 512`, produce the **actual chunks** with the same sliding-window logic as §3 (sentences as units, `MAX_TOKENS=512`, `OVERLAP=64`) and write them to a new CSV.

Output rows follow exactly the **corpus format** of `/mnt/data/vpitsilou/wisefood/new_experiments/new_corpus/corpus_top_journal.csv`:

| column | value |
|---|---|
| `chunk_id` | `paperId` (unsplit) or `paperId_0`, `paperId_1`, … (split) |
| `chunk_text` | chunk text |
| `type` | `abstract` |
| `chunk_metadata` | Python-dict literal string with keys `title, venue, year, referenceCount, citationCount, influentialCitationCount, authors, DOI` |

`INCLUDE_UNSPLIT = True` → abstracts ≤ 512 tokens are also written as single chunks so the CSV is a complete corpus; set to `False` to write only the split (>512) chunks.

In [ ]:
from tqdm.auto import tqdm

# ── Corpus format (same as corpus_top_journal.csv) ──
CHUNKS_OUTPUT_CSV = "../data/original/abstracts/abstract_chunks.csv"
INCLUDE_UNSPLIT = True   # False → write only the split (>512) chunks

def build_overlapping_chunk_texts(sentences: list[str]) -> list[str]:
    """
    Same sliding-window as estimate_split() in §3, but returns the actual
    chunk texts (sentences joined with ' ') instead of just counting.
    """
    n = len(sentences)
    if n == 0:
        return []
    token_counts = [count_tokens(s) for s in sentences]
    cumsum = [0] * (n + 1)
    for i in range(n):
        cumsum[i + 1] = cumsum[i] + token_counts[i]
    chunks: list[str] = []
    start = 0
    while start < n:
        # All remaining sentences fit in one chunk → emit once and stop.
        if cumsum[n] - cumsum[start] <= MAX_TOKENS:
            chunks.append(" ".join(sentences[start:n]))
            break
        # Expand end as far as possible within MAX_TOKENS.
        end = start
        while end < n and (cumsum[end + 1] - cumsum[start]) <= MAX_TOKENS:
            end += 1
        if end == start:
            # Single oversized sentence → emitted as one chunk.
            chunks.append(sentences[start])
            start += 1
            continue
        chunks.append(" ".join(sentences[start:end]))
        # Advance start for ~OVERLAP tokens (same logic as estimate_split).
        new_start = start
        for k in range(start + 1, end + 1):
            tail = cumsum[end] - cumsum[k]
            if tail >= OVERLAP:
                new_start = k
            else:
                break
        start = max(new_start, start + 1)
    return chunks

def make_chunk_metadata(row) -> dict:
    """chunk_metadata dict with the same keys as corpus_top_journal.csv."""
    doi = row.get("externalIds_DOI") or row.get("catalog_doi")
    if pd.notna(doi) and not str(doi).startswith(("http://", "https://")):
        doi = f"https://doi.org/{doi}"
    def _int(v):
        try:
            return int(v)
        except (TypeError, ValueError):
            return None
    return {
        "title":                    row.get("title"),
        "venue":                    row.get("venue"),
        "year":                     _int(row.get("year")),
        "referenceCount":           _int(row.get("referenceCount")),
        "citationCount":            _int(row.get("citationCount")),
        "influentialCitationCount": _int(row.get("influentialCitationCount")),
        "authors":                  row.get("authors"),
        "DOI":                      doi,
    }

chunk_rows = []
n_split_sources = 0
n_unsplit = 0
for _, row in tqdm(work.iterrows(), total=len(work), desc="Splitting abstracts > 512"):
    text = row["abstract"] if isinstance(row["abstract"], str) else ""
    token_count = count_tokens(text)
    paper_id = str(row["paperId"])
    if token_count <= MAX_TOKENS:
        n_unsplit += 1
        if not INCLUDE_UNSPLIT:
            continue
        chunks = [text]
    else:
        n_split_sources += 1
        chunks = build_overlapping_chunk_texts(split_into_sentences(text))
        if not chunks and text.strip():
            chunks = [text]
    metadata_str = str(make_chunk_metadata(row))
    for i, c in enumerate(chunks):
        # Use paperId as chunk_id. For split abstracts, append _{i}.
        chunk_id = paper_id if len(chunks) == 1 else f"{paper_id}_{i}"
        chunk_rows.append({
            "chunk_id":       chunk_id,
            "chunk_text":     c,
            "type":           "abstract",
            "chunk_metadata": metadata_str,
        })
chunks_df = pd.DataFrame(chunk_rows, columns=["chunk_id", "chunk_text", "type", "chunk_metadata"])
chunks_df.to_csv(CHUNKS_OUTPUT_CSV, index=False)
print(f"💾 Saved chunk corpus → {CHUNKS_OUTPUT_CSV}")
print(f"   split sources (>512 tokens): {n_split_sources:,}")
print(f"   unsplit sources (≤512)     : {n_unsplit:,}" + ("" if INCLUDE_UNSPLIT else "  (excluded from CSV)"))
print(f"   total chunks               : {len(chunks_df):,}")

## 5. Summary statistics

In [ ]:
total = len(counts_df)
need = int(counts_df["needs_split"].sum())
no_need = total - need

print("── Token-count summary (all abstracts) ──")
print(counts_df["abstract_token_count"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

print(f"\n── Split need (threshold: {MAX_TOKENS} tokens) ──")
print(f"  No split needed : {no_need:>10,}  ({no_need / total:.1%})")
print(f"  Split needed    : {need:>10,}  ({need / total:.1%})")

sub = counts_df[counts_df["needs_split"]]
if len(sub):
    total_chunks = int(sub["estimated_chunks"].sum())
    print(f"\n── Would-be chunks for the {need:,} abstracts needing a split ──")
    print(f"  Total chunks if split       : {total_chunks:,}")
    print(f"  Extra chunks vs. 1/abstract : {total_chunks - need:,}")
    print("  chunks per abstract:")
    print(sub["estimated_chunks"].describe(percentiles=[0.5, 0.9, 0.99]).to_string())

print("\n── Split need by abstract_quality ──")
byq = (
    counts_df
    .groupby("abstract_quality")
    .agg(n=("paperId", "size"), need_split=("needs_split", "sum"))
    .assign(pct_need=lambda d: (d["need_split"] / d["n"] * 100).round(1))
)
print(byq.to_string())

print("\n── Top 10 longest abstracts ──")
cols = ["paperId", "title", "abstract_quality", "abstract_token_count", "num_sentences", "estimated_chunks"]
print(counts_df.nlargest(10, "abstract_token_count")[cols].to_string(max_colwidth=70))

## 6. Save per-abstract counts (counts only — no chunks)

In [ ]:
OUTPUT_CSV = "../data/original/abstracts/abstract_chunk_counts.csv"

counts_df.to_csv(OUTPUT_CSV, index=False)
print(f"💾 Saved per-abstract chunk counts → {OUTPUT_CSV}")
print(f"   ({len(counts_df):,} rows × {len(counts_df.columns)} cols)")

## 7. Semantic clustering — configuration (CLI-arg compatible)

The whole clustering stage is written so it can run **either** interactively in this notebook **or** as a standalone script with CLI arguments — the `parse_args()` cell below is the single source of truth, and running the notebook cell is equivalent to running the script with the same defaults.

Key parameters:

| arg | default | meaning |
|---|---|---|
| `--chunks-csv` | `abstract_chunks.csv` (from §4b) | input chunk corpus |
| `--n-clusters` | `20` | number of semantic clusters |
| `--embed-model` | `BAAI/bge-large-en-v1.5` | SentenceTransformer model |
| `--device` | auto (`cuda` if available) | GPU/CPU for embedding |
| `--batch-size` | `64` | embedding batch size |
| `--embeddings-mmap` | `embeddings_chunks.dat` | memory-mapped, reusable embedding store |
| `--output-dir` | `chunks/abstracts/clusters` | per-cluster CSVs |
| `--random-state` | `42` | reproducibility |
| `--force-recompute` | off | ignore cached embeddings |

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

# ── Defaults (same values usable from the CLI) ──
DEFAULT_CHUNKS_CSV   = "../data/original/abstracts/abstract_chunks.csv"
DEFAULT_EMBED_MODEL  = "BAAI/bge-large-en-v1.5"
DEFAULT_EMBEDDINGS   = "../data/original/abstracts/embeddings_chunks.dat"
DEFAULT_OUTPUT_DIR   = "../data/chunks/abstracts/clusters"

def parse_args(argv=None):
    """Single config source — interactive (argv=[]) or CLI (`python this.py --n-clusters 50 --gpu`)."""
    p = argparse.ArgumentParser(
        description="Cluster abstract chunks with MiniBatchKMeans + normalized bge embeddings"
    )
    p.add_argument("--chunks-csv",      default=DEFAULT_CHUNKS_CSV, help="chunk corpus CSV from §4b")
    p.add_argument("--n-clusters",      type=int, default=20, help="number of semantic clusters")
    p.add_argument("--embed-model",     default=DEFAULT_EMBED_MODEL, help="SentenceTransformer model")
    p.add_argument("--device",          default=None, choices=["cuda", "cpu"], help="defaults to cuda if available")
    p.add_argument("--batch-size",      type=int, default=64, help="embedding batch size")
    p.add_argument("--embeddings-mmap", default=DEFAULT_EMBEDDINGS, help="memory-mapped embedding store (reusable)")
    p.add_argument("--output-dir",      default=DEFAULT_OUTPUT_DIR, help="per-cluster CSV output dir")
    p.add_argument("--random-state",    type=int, default=42)
    p.add_argument("--force-recompute", action="store_true", help="recompute embeddings even if cache exists")
    return p.parse_args(argv)

# Interactive default: same as running the script without arguments.
args = parse_args([])
print("⚙️  Config")
for k, v in sorted(vars(args).items()):
    print(f"   {k:>18}: {v}")

## 8. Embed chunks — normalized, GPU-aware, memory-mapped & reusable

- Encodes `chunk_text` with `BAAI/bge-large-en-v1.5` and **normalized embeddings** (`normalize_embeddings=True`) → cosine similarity becomes a plain dot product.
- **GPU support**: `--device cuda` (auto-selected when a GPU is available) → embeddings are computed on the GPU.
- **Memory-mapped reusable embeddings**: the matrix is persisted as a raw float32 `np.memmap` file plus a JSON sidecar (model, row count, dimension, and a sha1 signature of the chunk ids). On the next run the matrix is **memory-mapped directly with no recompute** as long as the signature matches — use `--force-recompute` to regenerate.

In [ ]:
import hashlib
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# ── Device selection (GPU if available) ──
if args.device is None:
    args.device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {args.device}" + (f" ({torch.cuda.get_device_name(0)})" if args.device == "cuda" else ""))

chunks_df = pd.read_csv(args.chunks_csv, low_memory=False)
print(f"✅ Loaded chunks: {len(chunks_df):,} rows from {args.chunks_csv}")

# bge-large-en-v1.5 → 1024-dim embeddings
EMBED_DIM   = 1024
EMBED_DTYPE = "float32"
mmap_path = Path(args.embeddings_mmap)
meta_path = Path(str(args.embeddings_mmap) + ".meta.json")

def _chunk_id_signature(df: pd.DataFrame) -> str:
    """sha1 over the ordered chunk ids — cache validity check."""
    return hashlib.sha1("\x1f".join(df["chunk_id"].astype(str)).encode()).hexdigest()

def _embedding_cache_valid() -> bool:
    if args.force_recompute or not (mmap_path.exists() and meta_path.exists()):
        return False
    try:
        meta = json.loads(meta_path.read_text())
    except (json.JSONDecodeError, OSError):
        return False
    return (
        meta.get("model") == args.embed_model
        and meta.get("n_rows") == len(chunks_df)
        and meta.get("dim") == EMBED_DIM
        and meta.get("signature") == _chunk_id_signature(chunks_df)
    )

if _embedding_cache_valid():
    print("♻️  Reusing cached memory-mapped embeddings (no recompute)")
    embeddings = np.memmap(
        mmap_path, dtype=EMBED_DTYPE, mode="r",
        shape=(len(chunks_df), EMBED_DIM),
    )
else:
    print(f"🧮 Computing embeddings with {args.embed_model} (normalized)…")
    model = SentenceTransformer(
        args.embed_model,
        cache_folder=str(HF_HUB_CACHE_DIR),
        device=args.device,
    )
    vectors = model.encode(
        chunks_df["chunk_text"].tolist(),
        batch_size=args.batch_size,
        normalize_embeddings=True,   # → cosine similarity = dot product
        convert_to_numpy=True,
        device=args.device,
        show_progress_bar=True,
    ).astype(EMBED_DTYPE)
    # Persist to a memory-mapped file (reusable across runs).
    mm = np.memmap(mmap_path, dtype=EMBED_DTYPE, mode="w+", shape=vectors.shape)
    mm[:] = vectors
    mm.flush()
    del mm, vectors
    meta_path.write_text(json.dumps({
        "model": args.embed_model,
        "n_rows": len(chunks_df),
        "dim": EMBED_DIM,
        "dtype": EMBED_DTYPE,
        "signature": _chunk_id_signature(chunks_df),
    }))
    print(f"💾 Embeddings saved → {mmap_path} ({os.path.getsize(mmap_path) / 1e9:.2f} GB, float32)")
    embeddings = np.memmap(
        mmap_path, dtype=EMBED_DTYPE, mode="r",
        shape=(len(chunks_df), EMBED_DIM),
    )
print(f"✅ Embeddings ready: shape={embeddings.shape} dtype={embeddings.dtype}")

## 9. Partition into clusters — scalable MiniBatchKMeans

`MiniBatchKMeans` is used instead of plain `KMeans` because it scales to large corpora (mini-batch updates, constant memory, `n_init="auto"`). Since the embeddings are **normalized**, cluster assignments reflect **cosine** semantic similarity. Each chunk receives **exactly one** cluster label — a non-overlapping partition.

In [ ]:
from sklearn.cluster import MiniBatchKMeans

n = len(chunks_df)
k = args.n_clusters
print(f"🔀 MiniBatchKMeans: n_clusters={k}, n_samples={n:,}, batch_size={min(args.batch_size, n)}")
km = MiniBatchKMeans(
    n_clusters=k,
    batch_size=min(args.batch_size, n),
    n_init="auto",
    random_state=args.random_state,
    verbose=0,
)
labels = km.fit_predict(embeddings)
chunks_df["cluster"] = labels
print(f"✅ Clustered {len(chunks_df):,} chunks into {km.n_clusters} clusters")
print(f"   inertia (sum of squared distances to centers): {km.inertia_:,.0f}")

## 10. Cluster statistics

Per-cluster: number of chunks, total/mean tokens, share of the corpus, and the top titles — plus a confirmation that the partition is **non-overlapping** (every chunk in exactly one cluster).

In [ ]:
import ast

def _token_len(series):
    return series.map(count_tokens)

clust_stats = (
    chunks_df
    .groupby("cluster")
    .agg(
        n_chunks     = ("chunk_id", "size"),
        total_tokens = ("chunk_text", lambda s: _token_len(s).sum()),
        mean_tokens  = ("chunk_text", lambda s: _token_len(s).mean().round(0)),
    )
    .assign(pct_corpus=lambda d: (d["n_chunks"] / len(chunks_df) * 100).round(2))
    .sort_values("n_chunks", ascending=False)
)
print("── Cluster statistics ──")
print(clust_stats.to_string())
print(f"\nTotal chunks: {len(chunks_df):,} | clusters: {len(clust_stats):,}")
print(f"Sizes: min={clust_stats['n_chunks'].min():,}, max={clust_stats['n_chunks'].max():,}")
# Titles per cluster (from chunk_metadata, same dict keys as corpus_top_journal.csv)
_titles = chunks_df["chunk_metadata"].map(
    lambda s: ast.literal_eval(s)["title"] if isinstance(s, str) and s.strip() else ""
)
print("\n── Top 3 titles per cluster ──")
for c in sorted(chunks_df["cluster"].unique()):
    tops = _titles[chunks_df["cluster"] == c].value_counts().head(3)
    print(f"  cluster {c:>3} | " + " | ".join(t[:60] for t in tops.index))

## 11. Save each cluster as a separate CSV + verification

- Each cluster → `cluster_000.csv`, `cluster_001.csv`, … under `--output-dir` (the cluster id lives in the filename).
- **All original columns preserved** — `chunk_id, chunk_text, type, chunk_metadata`, the same 4-column corpus format as `corpus_top_journal.csv`.
- **Verification**: every output file is read back and checked so that each input `chunk_id` appears in **exactly one** file — no duplicates, no missing rows, no unknown ids.

In [ ]:
from collections import Counter

OUTPUT_COLUMNS = ["chunk_id", "chunk_text", "type", "chunk_metadata"]   # corpus_top_journal.csv format
os.makedirs(args.output_dir, exist_ok=True)

written_paths = []
for c in sorted(chunks_df["cluster"].unique()):
    sub = chunks_df.loc[chunks_df["cluster"] == c, OUTPUT_COLUMNS]
    out = Path(args.output_dir) / f"abstracts_cluster_{c:02d}.csv"
    sub.to_csv(out, index=False)
    written_paths.append(out)
    print(f"💾 cluster {c:>3} → {out}  ({len(sub):,} rows)")

# ── Verification: every input row in exactly one output file ──
print("\n── Verification ──")
seen = Counter()
for p in written_paths:
    rb = pd.read_csv(p, dtype={"chunk_id": str})
    seen.update(rb["chunk_id"].tolist())
input_ids = set(chunks_df["chunk_id"].astype(str))
dupes   = {i for i, c in seen.items() if c > 1}
missing = input_ids - set(seen.keys())
extra   = set(seen.keys()) - input_ids
print(f"  output files : {len(written_paths)}")
print(f"  rows in files: {sum(seen.values()):,}")
print(f"  input rows   : {len(input_ids):,}")
print(f"  duplicates   : {len(dupes):,}")
print(f"  missing      : {len(missing):,}")
print(f"  unknown      : {len(extra):,}")
assert len(written_paths) == args.n_clusters, "expected one file per cluster"
assert sum(seen.values()) == len(input_ids), "row-count mismatch"
assert not dupes,   f"rows found in >1 file: {sorted(dupes)[:5]}"
assert not missing, f"rows missing: {sorted(missing)[:5]}"
assert not extra,   "rows in files that are not in the input"
print("✅ PASS — every input row belongs to exactly one output file")